### **Filtering and trimming messages**

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_core langgraph langchain_openai

In [2]:
import os, openai
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI

_ = load_dotenv(find_dotenv())
openai.api_key = os.environ['OPENAI_API_KEY']

llm_model = 'gpt-4o'
chat = ChatOpenAI(model = llm_model)

#### **Messages as state**

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage('Hi.', name = 'Bot', id = '1')]
messages.append(HumanMessage('Hi.', name = 'Lance', id = '2'))
messages.append(AIMessage('So you said you were researching ocean mammals?', name = 'Bot', id = '3'))
messages.append(HumanMessage('Yes, I know about whales. But what others should I learn about?', name = 'Lance', id = '4'))

for m in messages:
    m.pretty_print()
# By default, '.invoke()' doesnt print output
# However, in interactive environment, the last value of a cell is automatically displayed
# chat.invoke(messages).pretty_print()

In [ ]:
from IPython.display import Image, display
from langgraph.graph import MessagesState
from langgraph.graph import StateGraph, START, END

# Node
def chat_model_node(state: MessagesState):
    return {'messages': chat.invoke(state['messages'])}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node('chat_model', chat_model_node)
builder.add_edge(START, 'chat_model')
builder.add_edge('chat_model', END)
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

#### **Reducer**

In [ ]:
from langchain_core.messages import RemoveMessage

# Nodes
def filter_messages(state: MessagesState):
    # Delete all but the 2 most recent messages
    delete_messages = [RemoveMessage(id = m.id) for m in state['messages'][:-2]]
    return {'messages': delete_messages}

def chat_model_node(state: MessagesState):    
    return {'messages': [chat.invoke(state['messages'])]}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node('filter', filter_messages)
builder.add_node('chat_model', chat_model_node)
builder.add_edge(START, 'filter')
builder.add_edge('filter', 'chat_model')
builder.add_edge('chat_model', END)
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))
response = graph.invoke({'messages': messages})

#### **Filtering messages**

In [ ]:
# Node
def chat_model_node(state: MessagesState):
    return {'messages': [chat.invoke(state['messages'][-1:])]}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node('chat_model', chat_model_node)
builder.add_edge(START, 'chat_model')
builder.add_edge('chat_model', END)
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
messages.append(response['messages'][-1])
messages.append(HumanMessage(f'Tell me more about Narwhals!', name = 'Lance'))

# Invoke, using message filtering
response = graph.invoke({'messages': messages})

#### **Trim messages**
`trim_messages` limits the message history to a specified token count, ensuring the chat model processes only recent content within that limit. Unlike filtering, which selects a subset of past messages, trimming proactively restricts the context available to the model for generating responses.

In [ ]:
from langchain_core.messages import trim_messages

# Node
def chat_model_node(state: MessagesState):
    messages = trim_messages(
            state['messages'],
            max_tokens = 100,
            strategy = 'last',
            token_counter = ChatOpenAI(model = 'gpt-4o'),
            allow_partial = False,
        )
    return {"messages": [chat.invoke(messages)]}

# Build graph
builder = StateGraph(MessagesState)
builder.add_node("chat_model", chat_model_node)
builder.add_edge(START, "chat_model")
builder.add_edge("chat_model", END)
graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In [27]:
messages.append(response['messages'][-1])
messages.append(HumanMessage(f"Tell me where Orcas live!", name="Lance"))
response = graph.invoke({'messages': messages})